# LeetCode 767 — Reorganize String bằng Python

Link đề: https://leetcode.com/problems/reorganize-string/ · Độ khó: **Medium** · Chủ đề: Greedy, Hash Table, String, Counting, Sorting, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def reorganizeString(self, s: str) -> str:
        ...
```

Ghi chú: đề cho phép trả về **bất kỳ** cách sắp xếp hợp lệ nào, nên bài có **nhiều đáp án đúng**. Vì vậy ở phần test ta kiểm bằng **hàm kiểm tra tính hợp lệ** (`is_valid_reorg`) chứ không so với một chuỗi cứng.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: xếp lại các ký tự của chuỗi `s` sao cho **hai ký tự kề nhau không bao giờ giống nhau**; trả về một cách xếp bất kỳ, hoặc `""` nếu **không thể**.
2. Nắm các công cụ Python: `collections.Counter` và `most_common`, **hàng đợi ưu tiên** `heapq` (heap nhỏ nhất, mẹo dùng số **âm** để thành heap lớn nhất), gán theo bước nhảy `res[0::2]`, `itertools.permutations` để vét cạn, và công thức làm tròn lên `(n + 1) // 2`.
3. Đi từ **thử mọi hoán vị** → **heap tham lam (lấy ký tự đang nhiều nhất, "giữ chân" ký tự vừa dùng một lượt)** → **heap lấy hai ký tự mỗi lần** → **xếp vào vị trí chẵn rồi vị trí lẻ** → **quay lui có cắt tỉa bằng điều kiện khả thi**.
4. Hiểu mấu chốt: chuỗi sắp xếp được **khi và chỉ khi** ký tự xuất hiện nhiều nhất có tối đa `ceil(n / 2) = (n + 1) // 2` lần; và để xếp, ta luôn ưu tiên đặt ký tự **còn nhiều nhất** trước.
5. Tránh các lỗi kinh điển: tham lam theo **thứ tự chữ cái** thay vì theo **tần suất**, quên kiểm tra **trường hợp không khả thi** (trả chuỗi sai thay vì `""`), trả lại ký tự vừa dùng vào heap **ngay lập tức** (làm hai ký tự giống nhau kề nhau), và so khớp đáp án với một chuỗi cứng dù đề nhận nhiều đáp án.

## 2. Đề bài & ví dụ

Cho chuỗi `s`, hãy **sắp xếp lại các ký tự** của `s` sao cho **hai ký tự liền kề bất kỳ không giống nhau**.

Trả về **một cách sắp xếp hợp lệ bất kỳ**, hoặc trả về chuỗi rỗng `""` nếu không thể.

**Ví dụ 1**

- Input: `s = "aab"`
- Output: `"aba"`

**Ví dụ 2**

- Input: `s = "aaab"`
- Output: `""`
- Giải thích: có 3 chữ `a` trong chuỗi dài 4; dù xếp thế nào cũng có hai chữ `a` kề nhau.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `s = "aabbcc"` → một đáp án hợp lệ là `"abcabc"` (còn nhiều đáp án khác như `"acbacb"`, `"ababcc"` thì **không** hợp lệ vì `cc` kề nhau).

**Ràng buộc:**

- `1 <= s.length <= 500`.
- `s` chỉ gồm chữ cái tiếng Anh viết thường.

Chú ý: vì "bất kỳ cách sắp xếp hợp lệ nào" đều được chấp nhận, chuỗi mà code của bạn trả về có thể khác đáp án mẫu mà vẫn đúng.

In [1]:
from typing import List
from collections import Counter
from heapq import heapify, heappush, heappop
from itertools import permutations, product
import random
import time

examples = [
    ("aab", True),        # True: có đáp án (đáp án bất kỳ hợp lệ)
    ("aaab", False),      # False: không có đáp án, phải trả ""
    ("aabbcc", True),
]
for s, feasible in examples:
    print(repr(s), "-> có đáp án?" , feasible)

'aab' -> có đáp án? True
'aaab' -> có đáp án? False
'aabbcc' -> có đáp án? True


## 3. Kiến thức nền

### 3.1. `Counter` và `most_common`

`Counter(s)` đếm số lần mỗi ký tự xuất hiện, trả về đối tượng giống dict. `most_common()` trả list các cặp `(ký tự, số lần)` **sắp giảm dần theo số lần**. `max(cnt.values())` cho tần suất lớn nhất.

In [2]:
cnt = Counter("aaabbc")
print(cnt)
print(cnt.most_common())
print("tần suất lớn nhất:", max(cnt.values()), "| số ký tự khác nhau:", len(cnt))

Counter({'a': 3, 'b': 2, 'c': 1})
[('a', 3), ('b', 2), ('c', 1)]
tần suất lớn nhất: 3 | số ký tự khác nhau: 3


### 3.2. Hàng đợi ưu tiên `heapq` và mẹo số âm

`heapq` là **heap nhỏ nhất** trên một `list`: `heappush(h, x)` thêm, `heappop(h)` lấy phần tử **nhỏ nhất** ra, đều `O(log n)`; `heapify(h)` biến list thành heap trong `O(n)`. Muốn lấy phần tử **lớn nhất** (ở đây: ký tự còn nhiều nhất), ta lưu **số đếm âm**: `(-số_lần, ký_tự)`; phần tử nhỏ nhất là số âm "lớn nhất về độ lớn". Khi so sánh các tuple, Python so từng phần tử từ trái sang: nếu `-số_lần` bằng nhau thì so tiếp ký tự.

In [3]:
cnt = Counter("aaabbc")
heap = [(-c, ch) for ch, c in cnt.items()]
heapify(heap)
print("heap:", heap)
print("lấy ra:", heappop(heap))         # (-3, 'a'): ký tự nhiều nhất
heappush(heap, (-1, "a"))
print("lấy ra:", heappop(heap))         # (-2, 'b')
print("còn lại:", sorted(heap))

heap: [(-3, 'a'), (-2, 'b'), (-1, 'c')]
lấy ra: (-3, 'a')
lấy ra: (-2, 'b')
còn lại: [(-1, 'a'), (-1, 'c')]


### 3.3. Gán theo bước nhảy: vị trí chẵn rồi vị trí lẻ

Danh sách `res = [None] * n` cho phép ghi vào vị trí tùy ý. Các vị trí **chẵn** là `0, 2, 4, ...` (`range(0, n, 2)`), vị trí **lẻ** là `1, 3, 5, ...` (`range(1, n, 2)`). Hai ký tự ở hai vị trí **kề nhau** luôn có một vị trí chẵn và một vị trí lẻ; hai vị trí **cùng tính chẵn lẻ** không bao giờ kề nhau.

In [4]:
n = 7
print("chẵn:", list(range(0, n, 2)), "| lẻ:", list(range(1, n, 2)))
res = [None] * n
order = list(range(0, n, 2)) + list(range(1, n, 2))
print("thứ tự lấp:", order)
for i, ch in zip(order, "aaaabbc"):
    res[i] = ch
print("".join(res))

chẵn: [0, 2, 4, 6] | lẻ: [1, 3, 5]
thứ tự lấp: [0, 2, 4, 6, 1, 3, 5]
ababaca


### 3.4. Vét cạn bằng `itertools.permutations`

`permutations(s)` sinh mọi hoán vị (kể cả trùng lặp khi `s` có ký tự lặp), `n!` phần tử. Dùng `set(...)` để bỏ trùng. Chỉ dùng được cho `n` nhỏ (≤ 8 ký tự: `8! = 40320`).

In [5]:
perms = set("".join(p) for p in permutations("aab"))
print(sorted(perms), "| số hoán vị phân biệt:", len(perms))
print("số hoán vị (kể cả trùng) của 'aabbcc':", len(list(permutations("aabbcc"))))

['aab', 'aba', 'baa'] | số hoán vị phân biệt: 3
số hoán vị (kể cả trùng) của 'aabbcc': 720


### 3.5. Làm tròn lên và điều kiện `(n + 1) // 2`

`ceil(n / 2)` với số nguyên viết gọn là `(n + 1) // 2`: `n = 4 → 2`, `n = 5 → 3`, `n = 7 → 4`. Điều kiện khả thi của bài sẽ dùng chính con số này.

In [6]:
for n in range(1, 9):
    print(f"n = {n}: ceil(n/2) = {(n + 1) // 2}")

n = 1: ceil(n/2) = 1
n = 2: ceil(n/2) = 1
n = 3: ceil(n/2) = 2
n = 4: ceil(n/2) = 2
n = 5: ceil(n/2) = 3
n = 6: ceil(n/2) = 3
n = 7: ceil(n/2) = 4
n = 8: ceil(n/2) = 4


### 3.6. Hàm kiểm tra hai ký tự kề nhau khác nhau

Duyệt cặp kề: `all(t[i] != t[i + 1] for i in range(len(t) - 1))`. Một chuỗi rỗng hay một ký tự đều "không có cặp kề" nên thỏa. Ta sẽ dùng hàm này nhiều lần.

In [7]:
def no_adjacent_equal(t):
    return all(t[i] != t[i + 1] for i in range(len(t) - 1))

for t in ["aba", "aab", "", "a", "abab", "abba"]:
    print(repr(t), no_adjacent_equal(t))

'aba' True
'aab' False
'' True
'a' True
'abab' True
'abba' False


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Xét `s = "aab"` (hai chữ `a`, một chữ `b`). Có ba chỗ đặt: hai chữ `a` **không được kề nhau** nên cần một chữ khác chen giữa ⇒ `a b a`. Vừa đủ.

Xét `s = "aaab"`: ba chữ `a` cần ít nhất **hai** ký tự khác để ngăn cách (`a _ a _ a`), nhưng chỉ có một chữ `b` ⇒ **không thể**.

Tổng quát: nếu một ký tự xuất hiện `m` lần thì cần ít nhất `m - 1` ký tự khác ngăn cách chúng. Chuỗi có `n` ký tự, trong đó `n - m` ký tự khác: cần `n - m >= m - 1`, tức `m <= (n + 1) / 2`, hay `m <= (n + 1) // 2` với số nguyên.

Ngược lại, nếu **mọi** ký tự đều xuất hiện không quá `(n + 1) // 2` lần thì **luôn xếp được** — bằng cách đặt ký tự **nhiều nhất** vào các vị trí chẵn `0, 2, 4, ...` trước. Hãy thử một ví dụ cụ thể bằng tay: `s = "aaabbc"` (`a`: 3, `b`: 2, `c`: 1; `n = 6`, ngưỡng `(6 + 1) // 2 = 3`).

- Vị trí chẵn `0, 2, 4` ⇐ ba chữ `a`: `a _ a _ a _`.
- Còn các vị trí lẻ `1, 3, 5` ⇐ `b, b, c`: `a b a b a c`.

Kiểm: `abab ac`... không có hai ký tự kề nhau giống nhau ✓.

Hãy kiểm bằng code điều kiện `max <= (n + 1) // 2` trên mọi chuỗi nhỏ so với vét cạn.

In [8]:
def feasible_formula(s):
    return max(Counter(s).values()) <= (len(s) + 1) // 2

def feasible_bruteforce(s):
    return any(no_adjacent_equal("".join(p)) for p in set(permutations(s)))

total = 0
for n in range(1, 8):
    for tup in product("abc", repeat=n):
        s = "".join(tup)
        assert feasible_formula(s) == feasible_bruteforce(s), s
        total += 1
print("công thức max <= (n+1)//2 khớp vét cạn trên", total, "chuỗi 'abc' độ dài <= 7")
for s in ["aab", "aaab", "aaabbc", "aabbcc", "aaaabbb"]:
    print(f"{s}: max = {max(Counter(s).values())}, ngưỡng = {(len(s) + 1) // 2} -> khả thi? {feasible_formula(s)}")

công thức max <= (n+1)//2 khớp vét cạn trên 3279 chuỗi 'abc' độ dài <= 7
aab: max = 2, ngưỡng = 2 -> khả thi? True
aaab: max = 3, ngưỡng = 2 -> khả thi? False
aaabbc: max = 3, ngưỡng = 3 -> khả thi? True
aabbcc: max = 2, ngưỡng = 3 -> khả thi? True
aaaabbb: max = 4, ngưỡng = 4 -> khả thi? True


## 5. Cách 1 — Brute Force: thử mọi hoán vị

**Ý tưởng:** sinh **mọi hoán vị** của các ký tự, kiểm tra từng hoán vị xem có hai ký tự kề nhau giống nhau không; gặp hoán vị hợp lệ đầu tiên thì trả về, không có thì trả `""`.

Chắc chắn đúng, nên ta dùng làm **oracle** về tính khả thi (đáp án tồn tại hay không) trên chuỗi ngắn.

In [9]:
def reorganize_bruteforce(s: str) -> str:
    for p in sorted(set(permutations(s))):
        t = "".join(p)
        if no_adjacent_equal(t):
            return t
    return ""

for s, feasible in examples:
    out = reorganize_bruteforce(s)
    print(repr(s), "->", repr(out))

'aab' -> 'aba'
'aaab' -> ''
'aabbcc' -> 'abacbc'


**Độ phức tạp:** `n!` hoán vị, mỗi cái kiểm `O(n)` ⇒ `O(n · n!)` thời gian; bộ nhớ `O(n · n!)` nếu giữ tập hoán vị. Với `n = 500` là hoàn toàn không thể.

**Nút thắt:** số hoán vị bùng nổ. Ta thấy ngay ở bảng dưới: thêm một ký tự là thời gian nhân lên cả chục lần.

In [10]:
rng = random.Random(1)
for n in [5, 6, 7, 8]:
    s = "".join(rng.choice("abcd") for _ in range(n))
    t0 = time.perf_counter()
    out = reorganize_bruteforce(s)
    dt = time.perf_counter() - t0
    print(f"n = {n}: s = {s!r:10s} kết quả {out!r:10s} ~{dt * 1000:8.1f} ms")

n = 5: s = 'bacad'    kết quả 'abacd'    ~     0.0 ms
n = 6: s = 'dddbad'   kết quả ''         ~     0.1 ms
n = 7: s = 'addadcb'  kết quả 'abdadcd'  ~     0.6 ms
n = 8: s = 'acaaaadb' kết quả ''         ~     2.7 ms


## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét cho ta một thuật toán tuyến tính:

1. **Điều kiện khả thi:** xếp được ⇔ `max_count <= (n + 1) // 2`.
2. **Tham lam theo tần suất:** ký tự còn **nhiều nhất** là ký tự "khó xếp nhất", nên ta **xếp nó trước**, và ở mỗi bước chọn ký tự còn nhiều nhất **khác ký tự vừa đặt**.

Có hai cách hiện thực ý "ưu tiên ký tự nhiều nhất, khác ký tự trước":

- **Heap:** luôn lấy ký tự đứng đầu heap (đếm lớn nhất). Ký tự vừa dùng bị **giữ lại một lượt** (chưa trả vào heap) để lượt sau không chọn lại nó.
- **Xếp theo vị trí:** sắp ký tự theo tần suất giảm dần, lần lượt đặt vào các vị trí **chẵn** `0, 2, 4, ...` rồi sang các vị trí **lẻ** `1, 3, 5, ...`. Ký tự nhiều nhất chiếm các vị trí chẵn đầu tiên; vì nó không quá `(n + 1) // 2` lần nên **không bao giờ lấn sang vị trí lẻ kề với chính nó**.

Ví dụ `"aaabbc"`: sắp theo tần suất `aaabbc`; vị trí chẵn `0, 2, 4` ⇐ `a, a, a`; vị trí lẻ `1, 3, 5` ⇐ `b, b, c` ⇒ `a b a b a c`.

## 7. Các cách tiếp theo

Bốn cách dưới đây cùng dựa trên tham lam theo tần suất nhưng **cách thực hiện khác nhau**: heap một phần tử với "giữ chân", heap lấy hai phần tử, xếp theo vị trí chẵn/lẻ, và quay lui có cắt tỉa. Khi nộp LeetCode, chỉ đổi tên class thành `Solution`.

### 7.1. Cách 2 — Heap, lấy một ký tự và "giữ chân" ký tự vừa dùng

**Giải thích từng đoạn:**

1. Đếm tần suất; đưa `(-số_lần, ký_tự)` vào heap.
2. Biến `prev` là ký tự **vừa dùng nhưng còn dư**, đang bị giữ ngoài heap.
3. Mỗi vòng: lấy ký tự đứng đầu heap (nhiều nhất), ghi vào kết quả; **trả `prev` về heap** (giờ nó không còn kề ký tự vừa chọn); nếu ký tự vừa chọn còn dư thì giữ nó làm `prev` mới.
4. Hết heap mà `prev` vẫn còn ⇒ còn ký tự lẻ loi không thể xếp ⇒ trả `""`.

In [11]:
class SolutionHeapOne:
    def reorganizeString(self, s: str) -> str:
        heap = [(-c, ch) for ch, c in Counter(s).items()]
        heapify(heap)
        res = []
        prev = None                                   # (đếm âm, ký tự) đang bị giữ chân
        while heap:
            c, ch = heappop(heap)
            res.append(ch)
            if prev is not None:
                heappush(heap, prev)                  # trả lại ký tự lượt trước
                prev = None
            if c + 1 < 0:                             # ký tự này còn dư (đếm âm còn < 0)
                prev = (c + 1, ch)
        return "".join(res) if prev is None else ""

for s, feasible in examples:
    print(repr(s), "->", repr(SolutionHeapOne().reorganizeString(s)))

'aab' -> 'aba'
'aaab' -> ''
'aabbcc' -> 'abcabc'


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def reorganizeString(self, s: str) -> str:
        heap = [(-c, ch) for ch, c in Counter(s).items()]   # from collections import Counter
        heapify(heap)                                       # from heapq import heapify, heappush, heappop
        res = []
        prev = None
        while heap:
            c, ch = heappop(heap)
            res.append(ch)
            if prev is not None:
                heappush(heap, prev)
                prev = None
            if c + 1 < 0:
                prev = (c + 1, ch)
        return "".join(res) if prev is None else ""
```

**Độ phức tạp:** mỗi ký tự một lần `heappop` và tối đa một lần `heappush`, heap có tối đa 26 phần tử ⇒ `O(n log 26) = O(n)` thời gian; bộ nhớ `O(1)` (heap ≤ 26) cộng `O(n)` cho kết quả.

### 7.2. Cách 3 — Heap, mỗi lần lấy hai ký tự khác nhau

**Ý tưởng:** thay vì "giữ chân", mỗi bước lấy **hai ký tự nhiều nhất** (chắc chắn khác nhau vì mỗi ký tự là một phần tử riêng trong heap), ghi cả hai liền nhau, giảm đếm mỗi ký tự một đơn vị và trả lại heap nếu còn dư. Hai ký tự trong cùng một cặp tự khác nhau; ta còn cần chứng minh cặp **kế tiếp** không bắt đầu bằng ký tự cuối cặp trước — điều đó đúng vì ký tự thứ hai của cặp (đếm không lớn hơn ký tự thứ nhất) sau khi giảm vẫn **không lớn hơn** ký tự thứ nhất, nên lượt sau nó không thể đứng đầu heap trước ký tự thứ nhất (khi bằng nhau ta so thêm theo chữ cái, ký tự thứ nhất đã nhỏ hơn).

**Giải thích từng đoạn:**

1. Lặp khi heap còn **ít nhất hai** phần tử: lấy hai, ghi `a, b`, rồi trả lại phần còn dư.
2. Khi chỉ còn **một** phần tử: nếu đếm của nó `> 1` thì sẽ có hai ký tự giống nhau kề nhau ⇒ `""`; nếu bằng `1` thì ghi nốt.

In [12]:
class SolutionHeapPair:
    def reorganizeString(self, s: str) -> str:
        heap = [(-c, ch) for ch, c in Counter(s).items()]
        heapify(heap)
        res = []
        while len(heap) >= 2:
            c1, a = heappop(heap)
            c2, b = heappop(heap)
            res.append(a)
            res.append(b)
            if c1 + 1 < 0:
                heappush(heap, (c1 + 1, a))
            if c2 + 1 < 0:
                heappush(heap, (c2 + 1, b))
        if heap:
            c, a = heappop(heap)
            if -c > 1:
                return ""
            res.append(a)
        return "".join(res)

for s, feasible in examples:
    print(repr(s), "->", repr(SolutionHeapPair().reorganizeString(s)))

'aab' -> 'aba'
'aaab' -> ''
'aabbcc' -> 'abcabc'


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def reorganizeString(self, s: str) -> str:
        heap = [(-c, ch) for ch, c in Counter(s).items()]
        heapify(heap)
        res = []
        while len(heap) >= 2:
            c1, a = heappop(heap)
            c2, b = heappop(heap)
            res.append(a)
            res.append(b)
            if c1 + 1 < 0:
                heappush(heap, (c1 + 1, a))
            if c2 + 1 < 0:
                heappush(heap, (c2 + 1, b))
        if heap:
            c, a = heappop(heap)
            if -c > 1:
                return ""
            res.append(a)
        return "".join(res)
```

**Độ phức tạp:** `O(n log 26) = O(n)` thời gian; bộ nhớ `O(n)` cho kết quả, heap `O(1)`.

### 7.3. Cách 4 — Xếp vào vị trí chẵn rồi vị trí lẻ

**Giải thích từng đoạn:**

1. Đếm tần suất; nếu `max > (n + 1) // 2` trả `""` ngay.
2. Duyệt `cnt.most_common()` (**giảm dần theo tần suất**). Dùng con trỏ vị trí `i` bắt đầu từ `0`, mỗi lần đặt một ký tự thì `i += 2`.
3. Khi `i >= n` (hết vị trí chẵn), **quay lại vị trí `1`** (đầu các vị trí lẻ).
4. Kết quả: ký tự nhiều nhất chiếm các vị trí chẵn đầu tiên.

Vì sao hai ký tự giống nhau không bao giờ kề nhau? Lập luận gọn nằm ở mục 10 (một ký tự chỉ có thể "vắt" từ vị trí chẵn sang vị trí lẻ và chạm chính nó nếu nó có ít nhất `(n + 1) // 2` bản sao, nhưng khi đó nó phải là ký tự được đặt đầu tiên và nằm trọn ở các vị trí chẵn).

In [13]:
class SolutionEvenOdd:
    def reorganizeString(self, s: str) -> str:
        n = len(s)
        cnt = Counter(s)
        if max(cnt.values()) > (n + 1) // 2:
            return ""
        res = [None] * n
        i = 0
        for ch, c in cnt.most_common():
            for _ in range(c):
                if i >= n:
                    i = 1
                res[i] = ch
                i += 2
        return "".join(res)

for s, feasible in examples:
    print(repr(s), "->", repr(SolutionEvenOdd().reorganizeString(s)))

'aab' -> 'aba'
'aaab' -> ''
'aabbcc' -> 'abacbc'


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def reorganizeString(self, s: str) -> str:
        n = len(s)
        cnt = Counter(s)
        if max(cnt.values()) > (n + 1) // 2:
            return ""
        res = [None] * n
        i = 0
        for ch, c in cnt.most_common():
            for _ in range(c):
                if i >= n:
                    i = 1
                res[i] = ch
                i += 2
        return "".join(res)
```

**Độ phức tạp:** `most_common` sắp xếp tối đa 26 phần tử `O(26 log 26)`, mỗi ký tự đặt `O(1)` ⇒ `O(n)` thời gian; bộ nhớ `O(n)`.

### 7.4. Cách 5 — Quay lui có cắt tỉa bằng điều kiện khả thi

**Ý tưởng:** xây chuỗi từng ký tự một bằng **DFS (quay lui)**: ở mỗi vị trí thử các ký tự còn lại khác ký tự trước. Quay lui "ngây thơ" sẽ bùng nổ; ta **cắt tỉa** bằng điều kiện khả thi cho phần còn lại: sau khi đặt ký tự `ch`, phần còn lại (`left` ký tự, với ký tự đầu **không được** là `ch`) phải thỏa: `max <= (left + 1) // 2`, và nếu `left` lẻ mà ký tự nào đạt đúng `(left + 1) // 2` thì ký tự đó **bắt buộc đứng đầu** nên nó phải **khác** `ch`.

Nhờ cắt tỉa đúng, DFS **không bao giờ phải quay lui** khi bài khả thi.

**Giải thích từng đoạn:**

1. `can_finish(cnt, left, prev)` trả `True` nếu phần còn lại còn xếp được.
2. `dfs(prev, left)`: nếu `left == 0` thành công; ngược lại thử ký tự `ch` còn đếm `> 0`, `ch != prev`, đặt thử, nếu `can_finish` thì đệ quy; không được thì hoàn tác.
3. Thử ký tự theo thứ tự **đếm giảm dần**.

In [14]:
def can_finish(cnt, left, prev):
    """Phần còn lại (left ký tự, đếm cnt) có xếp được mà ký tự đầu khác prev không?"""
    if left == 0:
        return True
    mx = max(cnt.values())
    if mx > (left + 1) // 2:
        return False
    if left % 2 == 1 and mx == (left + 1) // 2:
        # ký tự đạt đúng ngưỡng bắt buộc phải đứng đầu -> không được trùng prev
        return all(ch != prev for ch, c in cnt.items() if c == mx)
    return True

class SolutionBacktrack:
    def reorganizeString(self, s: str) -> str:
        cnt = Counter(s)
        n = len(s)
        res = []

        def dfs(prev, left):
            if left == 0:
                return True
            for ch, c in sorted(cnt.items(), key=lambda kv: -kv[1]):
                if c > 0 and ch != prev:
                    cnt[ch] -= 1
                    res.append(ch)
                    if can_finish(cnt, left - 1, ch) and dfs(ch, left - 1):
                        return True
                    res.pop()
                    cnt[ch] += 1
            return False

        if not can_finish(cnt, n, None):
            return ""
        return "".join(res) if dfs(None, n) else ""

for s, feasible in examples:
    print(repr(s), "->", repr(SolutionBacktrack().reorganizeString(s)))

'aab' -> 'aba'
'aaab' -> ''
'aabbcc' -> 'abcabc'


**Code nộp LeetCode (Cách 5):**

```python
class Solution:
    def reorganizeString(self, s: str) -> str:
        cnt = Counter(s)
        n = len(s)
        res = []

        def can_finish(left, prev):
            if left == 0:
                return True
            mx = max(cnt.values())
            if mx > (left + 1) // 2:
                return False
            if left % 2 == 1 and mx == (left + 1) // 2:
                return all(ch != prev for ch, c in cnt.items() if c == mx)
            return True

        def dfs(prev, left):
            if left == 0:
                return True
            for ch, c in sorted(cnt.items(), key=lambda kv: -kv[1]):
                if c > 0 and ch != prev:
                    cnt[ch] -= 1
                    res.append(ch)
                    if can_finish(left - 1, ch) and dfs(ch, left - 1):
                        return True
                    res.pop()
                    cnt[ch] += 1
            return False

        if not can_finish(n, None):
            return ""
        return "".join(res) if dfs(None, n) else ""
```

**Độ phức tạp:** nhờ cắt tỉa hoàn hảo, mỗi vị trí chỉ thử vài ký tự và không quay lui: `O(n · 26 log 26)` ≈ `O(n)` thời gian; ngăn xếp đệ quy sâu `n` (tới 500 — dưới giới hạn mặc định 1000). Không cắt tỉa thì có thể `O(n!)`; ta so sánh số lời gọi dưới đây.

In [15]:
def count_dfs_calls(s, prune):
    cnt = Counter(s)
    calls = 0

    def dfs(prev, left):
        nonlocal calls
        calls += 1
        if left == 0:
            return True
        for ch in sorted(cnt):
            if cnt[ch] > 0 and ch != prev:
                cnt[ch] -= 1
                if (not prune or can_finish(cnt, left - 1, ch)) and dfs(ch, left - 1):
                    return True
                cnt[ch] += 1
        return False

    ok = dfs(None, len(s))
    return ok, calls

for s in ["aaaaabbc", "aaaaaabbcd", "aaaaaaabbcde", "aaaaaaaabbcdef"]:           # các chuỗi không khả thi (max > ngưỡng)
    ok1, c1 = count_dfs_calls(s, prune=False)
    ok2, c2 = count_dfs_calls(s, prune=True)
    print(f"{s}: khả thi? {ok1}/{ok2} | số lời gọi không cắt tỉa {c1:6d} | có cắt tỉa {c2}")

aaaaabbc: khả thi? False/False | số lời gọi không cắt tỉa     62 | có cắt tỉa 1
aaaaaabbcd: khả thi? False/False | số lời gọi không cắt tỉa    514 | có cắt tỉa 1
aaaaaaabbcde: khả thi? False/False | số lời gọi không cắt tỉa   5302 | có cắt tỉa 1
aaaaaaaabbcdef: khả thi? False/False | số lời gọi không cắt tỉa  65210 | có cắt tỉa 1


## 8. TRACE từng bước

Ta theo dõi cách 2 (heap một phần tử) trên `s = "aaabbc"`: in heap, ký tự được lấy, ký tự `prev` đang giữ chân, và chuỗi kết quả sau mỗi bước.

Bảng trạng thái (đếm của `a, b, c` = `3, 2, 1`):

| bước | heap trước (đếm, ký tự) | lấy | `prev` giữ | trả `prev` | kết quả |
|---|---|---|---|---|---|
| 1 | `a3, b2, c1` | `a` | `a2` | — | `a` |
| 2 | `b2, c1` | `b` | `b1` (sau khi trả `a2`) | `a2` vào heap | `ab` |
| 3 | `a2, c1` | `a` | `a1` (sau khi trả `b1`) | `b1` vào heap | `aba` |
| 4 | `b1, c1` | `b` | (hết, không giữ) | `a1` vào heap | `abab` |
| 5 | `a1, c1` | `a` | — | — | `ababa` |
| 6 | `c1` | `c` | — | — | `ababac` |

Hàm dưới in từng bước để bạn đối chiếu.

In [16]:
def trace_heap(s):
    heap = [(-c, ch) for ch, c in Counter(s).items()]
    heapify(heap)
    res = []
    prev = None
    step = 0
    print("heap ban đầu:", sorted(heap))
    while heap:
        step += 1
        before = sorted(heap)
        c, ch = heappop(heap)
        res.append(ch)
        returned = None
        if prev is not None:
            heappush(heap, prev)
            returned = prev
            prev = None
        if c + 1 < 0:
            prev = (c + 1, ch)
        print(f"bước {step}: heap trước = {before} | lấy {(-c, ch)} | trả lại {returned} | giữ chân {prev} | kết quả = {''.join(res)!r}")
    if prev is not None:
        print("còn ký tự giữ chân:", prev, "-> không thể xếp")
        return ""
    return "".join(res)

print("KẾT QUẢ:", repr(trace_heap("aaabbc")))
print()
print("KẾT QUẢ:", repr(trace_heap("aaab")))

heap ban đầu: [(-3, 'a'), (-2, 'b'), (-1, 'c')]
bước 1: heap trước = [(-3, 'a'), (-2, 'b'), (-1, 'c')] | lấy (3, 'a') | trả lại None | giữ chân (-2, 'a') | kết quả = 'a'
bước 2: heap trước = [(-2, 'b'), (-1, 'c')] | lấy (2, 'b') | trả lại (-2, 'a') | giữ chân (-1, 'b') | kết quả = 'ab'
bước 3: heap trước = [(-2, 'a'), (-1, 'c')] | lấy (2, 'a') | trả lại (-1, 'b') | giữ chân (-1, 'a') | kết quả = 'aba'
bước 4: heap trước = [(-1, 'b'), (-1, 'c')] | lấy (1, 'b') | trả lại (-1, 'a') | giữ chân None | kết quả = 'abab'
bước 5: heap trước = [(-1, 'a'), (-1, 'c')] | lấy (1, 'a') | trả lại None | giữ chân None | kết quả = 'ababa'
bước 6: heap trước = [(-1, 'c')] | lấy (1, 'c') | trả lại None | giữ chân None | kết quả = 'ababac'
KẾT QUẢ: 'ababac'

heap ban đầu: [(-3, 'a'), (-1, 'b')]
bước 1: heap trước = [(-3, 'a'), (-1, 'b')] | lấy (3, 'a') | trả lại None | giữ chân (-2, 'a') | kết quả = 'a'
bước 2: heap trước = [(-1, 'b')] | lấy (1, 'b') | trả lại (-2, 'a') | giữ chân None | kết quả = 'ab'
bướ

Quan sát ca `"aaab"`: sau khi dùng hết `b`, heap chỉ còn `a` nhưng `a` đang bị giữ chân ⇒ không còn gì để chen giữa ⇒ trả `""`.

Giờ trace cách 4 (chẵn/lẻ): in thứ tự ký tự theo tần suất, con trỏ `i`, và mảng kết quả từng bước.

In [17]:
def trace_even_odd(s):
    n = len(s)
    cnt = Counter(s)
    print(f"n = {n}, ngưỡng (n+1)//2 = {(n + 1) // 2}, tần suất = {cnt.most_common()}")
    if max(cnt.values()) > (n + 1) // 2:
        print("quá ngưỡng -> không khả thi")
        return ""
    res = [None] * n
    i = 0
    for ch, c in cnt.most_common():
        for _ in range(c):
            if i >= n:
                i = 1
                print("  (hết vị trí chẵn, quay lại vị trí 1)")
            res[i] = ch
            print(f"  đặt {ch!r} vào vị trí {i}: {['.' if x is None else x for x in res]}")
            i += 2
    return "".join(res)

print("KẾT QUẢ:", repr(trace_even_odd("aaabbc")))

n = 6, ngưỡng (n+1)//2 = 3, tần suất = [('a', 3), ('b', 2), ('c', 1)]
  đặt 'a' vào vị trí 0: ['a', '.', '.', '.', '.', '.']
  đặt 'a' vào vị trí 2: ['a', '.', 'a', '.', '.', '.']
  đặt 'a' vào vị trí 4: ['a', '.', 'a', '.', 'a', '.']
  (hết vị trí chẵn, quay lại vị trí 1)
  đặt 'b' vào vị trí 1: ['a', 'b', 'a', '.', 'a', '.']
  đặt 'b' vào vị trí 3: ['a', 'b', 'a', 'b', 'a', '.']
  đặt 'c' vào vị trí 5: ['a', 'b', 'a', 'b', 'a', 'c']
KẾT QUẢ: 'ababac'


## 9. Code nộp LeetCode

Ta chọn **Cách 4 (xếp vào vị trí chẵn rồi lẻ)**: ngắn, không cần heap, `O(n)`, và dễ chứng minh khi hiểu "ký tự nhiều nhất chiếm vị trí chẵn". Class `Solution` đúng chữ ký:

```python
class Solution:
    def reorganizeString(self, s: str) -> str:
        n = len(s)
        cnt = Counter(s)
        if max(cnt.values()) > (n + 1) // 2:
            return ""
        res = [None] * n
        i = 0
        for ch, c in cnt.most_common():
            for _ in range(c):
                if i >= n:
                    i = 1
                res[i] = ch
                i += 2
        return "".join(res)
```

In [18]:
class Solution:
    def reorganizeString(self, s: str) -> str:
        n = len(s)
        cnt = Counter(s)
        if max(cnt.values()) > (n + 1) // 2:
            return ""
        res = [None] * n
        i = 0
        for ch, c in cnt.most_common():
            for _ in range(c):
                if i >= n:
                    i = 1
                res[i] = ch
                i += 2
        return "".join(res)

def is_valid_reorg(s, out):
    """Hàm kiểm tra tính hợp lệ: nếu có đáp án thì out phải là hoán vị của s không có hai ký tự kề nhau giống nhau; nếu không có đáp án thì out == ''."""
    if not feasible_formula(s):
        return out == ""
    return len(out) == len(s) and sorted(out) == sorted(s) and no_adjacent_equal(out)

for s, feasible in examples:
    out = Solution().reorganizeString(s)
    assert is_valid_reorg(s, out), (s, out)
    print(repr(s), "->", repr(out), "hợp lệ")

'aab' -> 'aba' hợp lệ
'aaab' -> '' hợp lệ
'aabbcc' -> 'abacbc' hợp lệ


## 10. Vì sao đúng

**Điều kiện cần.** Nếu ký tự `x` xuất hiện `m` lần mà không có hai `x` kề nhau thì giữa mỗi cặp `x` liên tiếp có ít nhất một ký tự khác ⇒ cần `>= m - 1` ký tự khác ⇒ `n - m >= m - 1` ⇒ `m <= (n + 1) / 2`, tức `m <= (n + 1) // 2`.

**Điều kiện đủ (cách 4 xây được).** Giả sử `max <= (n + 1) // 2`. Cách 4 lấp các vị trí theo thứ tự `P = (0, 2, 4, ..., 1, 3, 5, ...)`, và ký tự được đặt theo **tần suất giảm dần**, mỗi ký tự chiếm một **đoạn liên tiếp** trong `P`. Có `(n + 1) // 2` vị trí chẵn và `n // 2` vị trí lẻ.

Hai vị trí kề nhau luôn gồm **một chẵn, một lẻ**. Do đó một ký tự chỉ có thể đứng ở hai vị trí kề nhau nếu đoạn của nó chứa **cả** vị trí chẵn **lẫn** vị trí lẻ, tức nó **vắt qua ranh giới** giữa phần chẵn và phần lẻ của `P`. Giả sử ký tự `x` vắt qua ranh giới: đoạn của nó gồm các vị trí chẵn từ `e` tới cuối (`e` là vị trí chẵn đầu tiên của nó) và các vị trí lẻ `1, 3, ..., 2t - 1` (`t >= 1`). Để có hai vị trí kề nhau cần một vị trí chẵn `j >= e` sát một vị trí lẻ `<= 2t - 1`, tức `e <= 2t`. Khi đó số bản sao của `x` là `(số vị trí chẵn từ e) + t >= (số vị trí chẵn từ 2t) + t = (số vị trí chẵn) = (n + 1) // 2`. Vì `x` có không quá `(n + 1) // 2` bản sao nên **đúng bằng** `(n + 1) // 2` và `e = 2t`... nhưng một ký tự có đúng `(n + 1) // 2` bản sao là ký tự **nhiều nhất**, nên nó được đặt **đầu tiên** (`e = 0`) và lấp **trọn** các vị trí chẵn mà không sang phần lẻ — mâu thuẫn với `t >= 1`. (Trường hợp `n` chẵn có hai ký tự cùng `n / 2` bản sao: ký tự đầu lấp hết vị trí chẵn, ký tự sau lấp hết vị trí lẻ, cũng không vắt qua ranh giới.) Vậy không có hai ký tự giống nhau kề nhau.

Thay vì chỉ tin vào lập luận, ta **kiểm tra toàn diện**: với **mọi** chuỗi trên bảng chữ cái `{a, b, c, d}` độ dài `<= 8` (và nhiều chuỗi lớn ngẫu nhiên), mỗi cách đều phải thỏa hàm `is_valid_reorg`. Đây cũng là kiểm chứng "điều kiện đủ" trong thực tế.

In [19]:
methods = {
    "heap1": lambda s: SolutionHeapOne().reorganizeString(s),
    "heap2": lambda s: SolutionHeapPair().reorganizeString(s),
    "evenodd": lambda s: SolutionEvenOdd().reorganizeString(s),
    "backtrack": lambda s: SolutionBacktrack().reorganizeString(s),
    "final": lambda s: Solution().reorganizeString(s),
}

total = 0
n_feasible = 0
for n in range(1, 9):
    for tup in product("abcd", repeat=n):
        s = "".join(tup)
        feas = feasible_formula(s)
        n_feasible += feas
        for name, fn in methods.items():
            out = fn(s)
            assert is_valid_reorg(s, out), (name, s, out)
        total += 1
print("đã kiểm", total, "chuỗi 'abcd' độ dài 1..8 (trong đó", n_feasible, "chuỗi xếp được); 5 cách đều hợp lệ")

đã kiểm 87380 chuỗi 'abcd' độ dài 1..8 (trong đó 78640 chuỗi xếp được); 5 cách đều hợp lệ


## 11. Time / Space Complexity

Gọi `n = len(s)`, bảng chữ cái có `k = 26` ký tự.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi hoán vị | `O(n · n!)` | `O(n · n!)` (với tập hoán vị) |
| 2. Heap giữ chân | `O(n log k)` ≈ `O(n)` | `O(k)` + kết quả `O(n)` |
| 3. Heap lấy hai | `O(n log k)` ≈ `O(n)` | `O(k)` + kết quả `O(n)` |
| 4. Chẵn / lẻ | `O(n + k log k)` ≈ `O(n)` | `O(n)` |
| 5. Quay lui cắt tỉa | `O(n · k log k)` ≈ `O(n)` (không quay lui khi cắt tỉa đúng) | `O(n)` (ngăn xếp đệ quy sâu `n`) |

Với `n = 500`, mọi cách `O(n)` chạy trong chưa tới 1 ms. Đo thực tế trên chuỗi độ dài 500.

In [20]:
rng = random.Random(11)
s500 = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(500))
print("n =", len(s500), "| tần suất lớn nhất:", max(Counter(s500).values()), "| ngưỡng:", (len(s500) + 1) // 2)
for name, fn in methods.items():
    t0 = time.perf_counter()
    for _ in range(50):
        out = fn(s500)
    dt = (time.perf_counter() - t0) / 50
    assert is_valid_reorg(s500, out)
    print(f"{name:10s} {dt * 1000:8.3f} ms / lần")

n = 500 | tần suất lớn nhất: 32 | ngưỡng: 250
heap1         0.189 ms / lần
heap2         0.187 ms / lần
evenodd       0.046 ms / lần
backtrack     2.578 ms / lần
final         0.043 ms / lần


## 12. Bộ test

Bài có **nhiều đáp án đúng**, nên ta kiểm bằng **hàm kiểm tra tính hợp lệ** `is_valid_reorg(s, out)` (không so chuỗi cứng): nếu `s` khả thi thì `out` phải là hoán vị của `s` không có hai ký tự kề nhau giống nhau; nếu không khả thi thì `out == ""`. Ta kiểm: ví dụ đề, edge case, phản ví dụ, đối chiếu **tính khả thi** của mọi cách với oracle vét cạn, ngẫu nhiên có seed (đúng ràng buộc `n <= 500`), các trường hợp biên về ngưỡng.

In [21]:
def check_all(s):
    results = {}
    for name, fn in methods.items():
        out = fn(s)
        assert is_valid_reorg(s, out), (name, s, out)
        results[name] = out
    # mọi cách phải đồng ý về TÍNH KHẢ THI (trả "" hay không)
    assert len({out == "" for out in results.values()}) == 1, (s, results)
    return results["final"]

# 1) Ví dụ đề
assert check_all("aab") != "" and check_all("aaab") == ""
print("ví dụ đề: OK | aab ->", check_all("aab"), "| aabbcc ->", check_all("aabbcc"))

# 2) Edge case
edge = ["a", "aa", "ab", "aaa", "aab", "abb", "aabb", "aaabb", "aaabbb", "aaaabb", "abcdefghijklmnopqrstuvwxyz", "zzzzzyyyyyxxxxx"]
for s in edge:
    out = check_all(s)
    print(f"{s!r:32s} -> {out!r}")
print("edge case: OK")

ví dụ đề: OK | aab -> aba | aabbcc -> abacbc
'a'                              -> 'a'
'aa'                             -> ''
'ab'                             -> 'ab'
'aaa'                            -> ''
'aab'                            -> 'aba'
'abb'                            -> 'bab'
'aabb'                           -> 'abab'
'aaabb'                          -> 'ababa'
'aaabbb'                         -> 'ababab'
'aaaabb'                         -> ''
'abcdefghijklmnopqrstuvwxyz'     -> 'anbocpdqerfsgthuivjwkxlymz'
'zzzzzyyyyyxxxxx'                -> 'zyzyzxzxzxyxyxy'
edge case: OK


In [22]:
# 3) Biên của ngưỡng: ký tự 'a' đúng bằng ngưỡng (xếp được) và vượt ngưỡng 1 (không xếp được)
for n in range(1, 40):
    thr = (n + 1) // 2
    ok_s = "a" * thr + "".join(random.Random(n).choice("bcd") for _ in range(n - thr))
    bad_s = "a" * (thr + 1) + "".join(random.Random(n).choice("bcd") for _ in range(max(0, n - thr - 1)))
    assert check_all(ok_s) != ""
    if len(bad_s) >= 1 and len(bad_s) >= 2:
        assert check_all(bad_s) == ""
print("biên ngưỡng (n = 1..39): OK")

# 4) Đối chiếu tính khả thi với oracle vét cạn (hoán vị) trên mọi chuỗi 'abc' độ dài <= 7
for n in range(1, 8):
    for tup in product("abc", repeat=n):
        s = "".join(tup)
        bf = reorganize_bruteforce(s)
        out = check_all(s)
        assert (bf == "") == (out == ""), (s, bf, out)
print("khả thi khớp vét cạn trên mọi chuỗi 'abc' độ dài <= 7: OK")

# 5) Phản ví dụ: nhiều đáp án khác nhau đều hợp lệ -> so chuỗi cứng là sai cách kiểm
answers = {fn("aabbcc") for fn in methods.values()}
print("các đáp án khác nhau của 'aabbcc':", sorted(answers))
assert all(is_valid_reorg("aabbcc", a) for a in answers)
assert not is_valid_reorg("aabbcc", "ababcc")

biên ngưỡng (n = 1..39): OK


khả thi khớp vét cạn trên mọi chuỗi 'abc' độ dài <= 7: OK
các đáp án khác nhau của 'aabbcc': ['abacbc', 'abcabc']


In [23]:
# 6) Ngẫu nhiên có seed, đúng ràng buộc (n <= 500, chữ thường)
rng = random.Random(767)
for _ in range(500):
    n = rng.randint(1, 30)
    alpha = "abcdefghijklmnopqrstuvwxyz"[: rng.randint(1, 6)]
    s = "".join(rng.choice(alpha) for _ in range(n))
    check_all(s)
print("ngẫu nhiên nhỏ: OK")

n_feasible = n_infeasible = 0
for _ in range(300):
    n = rng.randint(100, 500)
    k = rng.randint(2, 26)
    alpha = "abcdefghijklmnopqrstuvwxyz"[:k]
    weights = [rng.random() ** 2 for _ in range(k)]                 # tạo phân phối lệch để có cả ca khả thi lẫn không
    s = "".join(rng.choices(alpha, weights=weights, k=n))
    out = check_all(s)
    if out:
        n_feasible += 1
    else:
        n_infeasible += 1
print(f"ngẫu nhiên lớn: {n_feasible} chuỗi xếp được, {n_infeasible} chuỗi không xếp được, tất cả hợp lệ")

# 7) Ràng buộc lớn nhất n = 500: các dạng xấu
cases = {
    "250 a + 250 b": "a" * 250 + "b" * 250,
    "250 a + 250 chữ khác": "a" * 250 + "".join(rng.choice("bcdefg") for _ in range(250)),
    "251 a + 249 chữ khác (không khả thi)": "a" * 251 + "".join(rng.choice("bcdefg") for _ in range(249)),
    "26 chữ đều nhau": ("abcdefghijklmnopqrstuvwxyz" * 20)[:500],
    "500 chữ a": "a" * 500,
}
for name, s in cases.items():
    out = check_all(s)
    print(f"{name:40s} n = {len(s)} -> {'xếp được' if out else 'không xếp được'}")

ngẫu nhiên nhỏ: OK


ngẫu nhiên lớn: 263 chuỗi xếp được, 37 chuỗi không xếp được, tất cả hợp lệ
250 a + 250 b                            n = 500 -> xếp được
250 a + 250 chữ khác                     n = 500 -> xếp được
251 a + 249 chữ khác (không khả thi)     n = 500 -> không xếp được
26 chữ đều nhau                          n = 500 -> xếp được
500 chữ a                                n = 500 -> không xếp được


## 13. Các lỗi người mới hay gặp

### Lỗi 1: tham lam theo **thứ tự chữ cái** thay vì theo tần suất

"Mỗi bước lấy ký tự nhỏ nhất (theo bảng chữ cái) khác ký tự trước" nghe hợp lý nhưng **làm hỏng** khi ký tự cuối bảng chữ cái xuất hiện nhiều.

In [24]:
class WrongAlphabetGreedy:
    def reorganizeString(self, s: str) -> str:
        cnt = Counter(s)
        out, prev = [], ""
        for _ in range(len(s)):
            cands = [ch for ch in sorted(cnt) if cnt[ch] > 0 and ch != prev]
            if not cands:
                return ""
            ch = cands[0]                         # SAI: chọn ký tự nhỏ nhất theo chữ cái
            out.append(ch)
            cnt[ch] -= 1
            prev = ch
        return "".join(out)

print("sai :", repr(WrongAlphabetGreedy().reorganizeString("abb")), "(đúng: có đáp án, ví dụ 'bab')")
print("đúng:", repr(Solution().reorganizeString("abb")))

sai : '' (đúng: có đáp án, ví dụ 'bab')
đúng: 'bab'


**Vì sao sai:** với `"abb"`, lấy `a` trước rồi `b`, rồi còn `b` nhưng `prev = "b"` ⇒ kẹt. Ký tự `b` nhiều nhất phải được đặt **trước**. **Cách sửa:** ưu tiên ký tự còn **nhiều nhất** (heap hoặc `most_common`).

### Lỗi 2: sắp xếp theo thứ tự chữ cái trước khi lấp chẵn/lẻ

In [25]:
class WrongSortedAlphabet:
    def reorganizeString(self, s: str) -> str:
        n = len(s)
        if max(Counter(s).values()) > (n + 1) // 2:
            return ""
        t = sorted(s)                             # SAI: sắp theo chữ cái, không theo tần suất
        res = [None] * n
        order = list(range(0, n, 2)) + list(range(1, n, 2))
        for i, ch in zip(order, t):
            res[i] = ch
        return "".join(res)

print("sai :", repr(WrongSortedAlphabet().reorganizeString("abb")), "(có hai chữ b kề nhau)")
print("đúng:", repr(Solution().reorganizeString("abb")))

sai : 'abb' (có hai chữ b kề nhau)
đúng: 'bab'


**Vì sao sai:** ký tự nhiều nhất phải là ký tự được lấp **đầu tiên** để nó chiếm trọn vị trí chẵn. Nếu nó đứng giữa thứ tự, nó có thể bị chia đôi và hai nửa chạm nhau. **Cách sửa:** sắp theo tần suất giảm dần (`most_common`).

### Lỗi 3: quên kiểm tra trường hợp không khả thi

In [26]:
class WrongNoCheck:
    def reorganizeString(self, s: str) -> str:
        n = len(s)
        res = [None] * n
        i = 0
        for ch, c in Counter(s).most_common():
            for _ in range(c):
                if i >= n:
                    i = 1
                res[i] = ch
                i += 2
        return "".join(res)                        # SAI: không kiểm max <= (n + 1) // 2

print("sai :", repr(WrongNoCheck().reorganizeString("aaab")), "(đúng phải là chuỗi rỗng)")
print("đúng:", repr(Solution().reorganizeString("aaab")))

sai : 'aaab' (đúng phải là chuỗi rỗng)
đúng: ''


**Vì sao sai:** khi một ký tự quá nhiều, cách lấp chẵn/lẻ vẫn trả ra một chuỗi — nhưng chuỗi đó có hai ký tự giống nhau kề nhau. **Cách sửa:** kiểm tra `max(cnt.values()) > (n + 1) // 2` trước và trả `""`.

### Lỗi 4: trả ký tự vừa dùng vào heap **ngay lập tức**

In [27]:
class WrongPushBack:
    def reorganizeString(self, s: str) -> str:
        heap = [(-c, ch) for ch, c in Counter(s).items()]
        heapify(heap)
        res = []
        while heap:
            c, ch = heappop(heap)
            res.append(ch)
            if c + 1 < 0:
                heappush(heap, (c + 1, ch))        # SAI: trả lại ngay, lượt sau có thể lấy lại chính nó
        return "".join(res)

print("sai :", repr(WrongPushBack().reorganizeString("aab")), "(hai chữ a kề nhau)")
print("đúng:", repr(Solution().reorganizeString("aab")))

sai : 'aab' (hai chữ a kề nhau)
đúng: 'aba'


**Vì sao sai:** ký tự vừa chọn còn đếm lớn nhất nên lại đứng đầu heap ở lượt sau. **Cách sửa:** **giữ chân** nó một lượt (chưa trả vào heap) như cách 2, hoặc lấy hai ký tự mỗi lần như cách 3.

### Lỗi 5: so khớp đáp án với một chuỗi cứng

Đề nhận **mọi** cách sắp hợp lệ. Với `"aabbcc"` có hàng chục đáp án đúng; nếu bạn viết kiểm tra `out == "abcabc"` thì các đáp án đúng khác như `"acbacb"` sẽ bị báo sai. Hãy dùng `is_valid_reorg`.

In [28]:
for cand in ["abcabc", "acbacb", "ababcc", "cbacba"]:
    print(f"{cand}: hợp lệ? {is_valid_reorg('aabbcc', cand)}  (so với 'abcabc' bằng ==: {cand == 'abcabc'})")

abcabc: hợp lệ? True  (so với 'abcabc' bằng ==: True)
acbacb: hợp lệ? True  (so với 'abcabc' bằng ==: False)
ababcc: hợp lệ? False  (so với 'abcabc' bằng ==: False)
cbacba: hợp lệ? True  (so với 'abcabc' bằng ==: False)


**Vì sao quan trọng:** hai chuỗi khác nhau đều hợp lệ nhưng `==` chỉ chấp nhận một. **Cách sửa:** kiểm tính hợp lệ (hoán vị + không kề nhau giống nhau).

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra bài "xếp lại sao cho không kề nhau giống nhau":**

- Có điều kiện về **hàng xóm** của phần tử và đề cho phép **sắp xếp lại tùy ý** ⇒ nghĩ ngay tới **đếm tần suất** và **điều kiện khả thi** `max <= ceil(n / 2)`.
- Cần "chọn phần tử khó xếp nhất trước" ⇒ **tham lam + heap** (ưu tiên đếm lớn nhất).
- Muốn giải không dùng heap ⇒ **lấp vị trí chẵn rồi lẻ** theo tần suất giảm dần.

**Mẹo:**

- Bài có nhiều đáp án đúng ⇒ viết **hàm kiểm tra tính hợp lệ** để test (hoán vị đúng + ràng buộc).
- Heap Python là heap nhỏ nhất ⇒ dùng số âm cho "lớn nhất trước".
- Mẫu "giữ chân một lượt" (cooldown) còn dùng cho các bài như Task Scheduler, Rearrange String k Distance Apart.

**Họ bài liên quan:**

| Bài | Ý tưởng |
|---|---|
| 358. Rearrange String k Distance Apart | tổng quát: hai ký tự giống nhau cách ít nhất `k` vị trí (heap + hàng đợi chờ) |
| 621. Task Scheduler | thời gian nguội giữa các lần chạy cùng tác vụ |
| 1054. Distant Barcodes | giống hệt bài này nhưng với mảng số nguyên |
| 984. String Without AAA or BBB | xếp `a`/`b` không có ba ký tự liên tiếp giống nhau |
| 1405. Longest Happy String | tham lam có giới hạn ba ký tự liên tiếp |

## 15. Biến thể

### 15.1. Chuỗi nhỏ nhất theo thứ tự từ điển (lexicographically smallest)

Đề gốc nhận bất kỳ đáp án, nhưng nếu hỏi chuỗi hợp lệ **nhỏ nhất theo từ điển** thì sao? Tham lam từng vị trí: thử ký tự nhỏ nhất `ch != prev` mà **phần còn lại vẫn khả thi** (dùng chính `can_finish` ở cách 5). Kiểm bằng vét cạn trên chuỗi ngắn.

In [29]:
def lexicographically_smallest(s):
    cnt = Counter(s)
    n = len(s)
    if not can_finish(cnt, n, None):
        return ""
    out, prev = [], None
    for left in range(n, 0, -1):
        for ch in sorted(cnt):
            if cnt[ch] > 0 and ch != prev:
                cnt[ch] -= 1
                if can_finish(cnt, left - 1, ch):
                    out.append(ch)
                    prev = ch
                    break
                cnt[ch] += 1
    return "".join(out)

def smallest_bruteforce(s):
    valid = [t for t in set("".join(p) for p in permutations(s)) if no_adjacent_equal(t)]
    return min(valid) if valid else ""

for n in range(1, 8):
    for tup in product("abc", repeat=n):
        s = "".join(tup)
        assert lexicographically_smallest(s) == smallest_bruteforce(s), s
print("nhỏ nhất theo từ điển khớp vét cạn trên mọi chuỗi 'abc' độ dài <= 7")
for s in ["aab", "aabbcc", "bbbaac", "zzzyyx", "aaab"]:
    print(f"{s!r:10s} -> {lexicographically_smallest(s)!r}")

nhỏ nhất theo từ điển khớp vét cạn trên mọi chuỗi 'abc' độ dài <= 7
'aab'      -> 'aba'
'aabbcc'   -> 'abacbc'
'bbbaac'   -> 'ababcb'
'zzzyyx'   -> 'xzyzyz'
'aaab'     -> ''


### 15.2. Distant Barcodes — cùng bài với mảng số nguyên (LeetCode 1054)

Thay ký tự bằng số nguyên (mã vạch). Thuật toán chẵn/lẻ y hệt, chỉ cần `Counter` trên số. Test bằng hàm kiểm tra tính hợp lệ (đề đảm bảo luôn có đáp án).

In [30]:
def distant_barcodes(barcodes):
    n = len(barcodes)
    cnt = Counter(barcodes)
    res = [None] * n
    i = 0
    for v, c in cnt.most_common():
        for _ in range(c):
            if i >= n:
                i = 1
            res[i] = v
            i += 2
    return res

def valid_barcodes(src, out):
    return sorted(out) == sorted(src) and all(out[i] != out[i + 1] for i in range(len(out) - 1))

rng = random.Random(152)
for _ in range(500):
    n = rng.randint(1, 40)
    k = rng.randint(2, 8)
    arr = [rng.randint(1, k) for _ in range(n)]
    if max(Counter(arr).values()) > (n + 1) // 2:
        continue                                      # đề đảm bảo có đáp án
    assert valid_barcodes(arr, distant_barcodes(arr)), arr
print([1, 1, 1, 2, 2, 2], "->", distant_barcodes([1, 1, 1, 2, 2, 2]))
print([1, 1, 1, 1, 2, 2, 3, 3], "->", distant_barcodes([1, 1, 1, 1, 2, 2, 3, 3]))

[1, 1, 1, 2, 2, 2] -> [1, 2, 1, 2, 1, 2]
[1, 1, 1, 1, 2, 2, 3, 3] -> [1, 2, 1, 2, 1, 3, 1, 3]


### 15.3. Đếm số cách sắp xếp hợp lệ phân biệt

Với `s` ngắn, **có bao nhiêu chuỗi phân biệt** thỏa điều kiện? Quy hoạch động / đệ quy có nhớ trên trạng thái `(đếm còn lại, ký tự cuối)`; kiểm bằng vét cạn hoán vị.

In [31]:
from functools import lru_cache

def count_arrangements(s):
    letters = sorted(set(s))
    start = tuple(Counter(s)[ch] for ch in letters)

    @lru_cache(maxsize=None)
    def go(counts, last):
        if sum(counts) == 0:
            return 1
        total = 0
        for j, c in enumerate(counts):
            if c > 0 and j != last:
                nxt = list(counts)
                nxt[j] -= 1
                total += go(tuple(nxt), j)
        return total

    return go(start, -1)

def count_bruteforce(s):
    return sum(1 for t in set("".join(p) for p in permutations(s)) if no_adjacent_equal(t))

for n in range(1, 8):
    for tup in product("abc", repeat=n):
        s = "".join(tup)
        assert count_arrangements(s) == count_bruteforce(s), s
print("số cách khớp vét cạn. Ví dụ:", {s: count_arrangements(s) for s in ["aab", "aabb", "aabbcc", "abcde"]})

số cách khớp vét cạn. Ví dụ: {'aab': 1, 'aabb': 2, 'aabbcc': 30, 'abcde': 120}


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi hoán vị | vét cạn `n!`, kiểm kề nhau | `O(n · n!)` | lớn | Chỉ làm oracle cho chuỗi ngắn |
| 2. Heap giữ chân | lấy ký tự nhiều nhất, giữ lại một lượt | `O(n)` | `O(1)` heap + kết quả | Mẫu "cooldown", mở rộng được cho khoảng cách `k` |
| 3. Heap lấy hai | mỗi bước ghi hai ký tự nhiều nhất khác nhau | `O(n)` | `O(1)` heap + kết quả | Khi muốn tránh biến `prev` |
| 4. Chẵn / lẻ | tần suất giảm dần, lấp `0, 2, 4, ..., 1, 3, ...` | `O(n)` | `O(n)` | **Nộp bài**: ngắn, không heap |
| 5. Quay lui cắt tỉa | thử ký tự, cắt bằng điều kiện khả thi | `O(n)` (không quay lui) | `O(n)` ngăn xếp | Khi cần biến thể có thêm ràng buộc (ví dụ nhỏ nhất theo từ điển) |

Cả bốn cách tuyến tính đều tạo ra các đáp án **khác nhau** (đều hợp lệ); ta chỉ cần `is_valid_reorg` để chấm.

In [32]:
for s in ["aabbcc", "aaabbc", "aabbbcdd"]:
    print(f"{s!r}:")
    for name, fn in methods.items():
        out = fn(s)
        print(f"   {name:10s} -> {out!r:12s} hợp lệ? {is_valid_reorg(s, out)}")

'aabbcc':
   heap1      -> 'abcabc'     hợp lệ? True
   heap2      -> 'abcabc'     hợp lệ? True
   evenodd    -> 'abacbc'     hợp lệ? True
   backtrack  -> 'abcabc'     hợp lệ? True
   final      -> 'abacbc'     hợp lệ? True
'aaabbc':
   heap1      -> 'ababac'     hợp lệ? True
   heap2      -> 'ababac'     hợp lệ? True
   evenodd    -> 'ababac'     hợp lệ? True
   backtrack  -> 'ababac'     hợp lệ? True
   final      -> 'ababac'     hợp lệ? True
'aabbbcdd':
   heap1      -> 'babdabcd'   hợp lệ? True
   heap2      -> 'babdabcd'   hợp lệ? True
   evenodd    -> 'babdbdac'   hợp lệ? True
   backtrack  -> 'babdabcd'   hợp lệ? True
   final      -> 'babdbdac'   hợp lệ? True


## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Các cell `TODO` chỉ **định nghĩa** hàm, không tự gọi, để notebook luôn chạy sạch.

### Bài 1 (đoán output)

Không chạy máy, cho biết mỗi chuỗi sau **có xếp được** không (và ngưỡng `(n + 1) // 2` là bao nhiêu):

```python
"aabbb"      # ?
"aaabbbc"    # ?
"abcabcabc"  # ?
"aaaabc"     # ?
```

### Bài 2 (tính tay)

Với `s = "aabbbcc"` hãy áp dụng **cách chẵn/lẻ** bằng tay: sắp theo tần suất giảm dần, ghi các vị trí `0..6` được lấp theo thứ tự nào, và viết chuỗi kết quả.

### Bài 3 (điền TODO)

Hoàn thành hàm lấp chẵn/lẻ (hai chỗ `TODO`: điều kiện không khả thi và chỗ quay lại vị trí lẻ).

In [33]:
def reorganize_fill_todo(s):
    n = len(s)
    cnt = Counter(s)
    # TODO: nếu ký tự nhiều nhất vượt ngưỡng thì trả ""
    res = [None] * n
    i = 0
    for ch, c in cnt.most_common():
        for _ in range(c):
            # TODO: nếu i đã vượt quá cuối mảng thì quay lại vị trí lẻ đầu tiên
            res[i] = ch
            i += 2
    return "".join(res)
# Không gọi reorganize_fill_todo ở đây. Hãy sửa rồi tự gọi ở một cell mới.

### Bài 4 (tự viết lại)

Không nhìn lại lời giải, hãy viết `reorganize_heap(s)` theo **Cách 2**: heap `(-đếm, ký tự)`, mỗi vòng lấy ký tự nhiều nhất, giữ chân ký tự vừa dùng một lượt, trả `""` nếu cuối cùng còn ký tự bị giữ.

In [34]:
def reorganize_heap(s):
    # TODO: tự viết
    return None
# Không gọi hàm ở đây.

### Bài 5 (mini challenge)

Viết `reorganize_k_apart(s, k)` (LeetCode 358): sắp xếp lại `s` sao cho **hai ký tự giống nhau cách nhau ít nhất `k` vị trí** (tức giữa chúng có ít nhất `k - 1` ký tự khác); trả `""` nếu không thể. Với `k = 2` đây chính là bài hiện tại. Kiểm chứng bằng hàm hợp lệ `is_valid_k` và so tính khả thi với vét cạn trên chuỗi ngắn.

In [35]:
def reorganize_k_apart(s, k):
    # TODO: tự viết
    return None
# Không gọi hàm ở đây.

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: đếm tần suất lớn nhất và so với `(n + 1) // 2`.
- Mức 2: `"aabbb"` có `n = 5`, ngưỡng `3`, `b` xuất hiện `3` lần; `"aaaabc"` có `n = 6`, ngưỡng `3`, `a` xuất hiện `4` lần.
- Mức 3: `"aabbb"` xếp được (`babab`); `"aaabbbc"` xếp được (`n = 7`, ngưỡng 4, max 3); `"abcabcabc"` xếp được; `"aaaabc"` **không** xếp được (`4 > 3`).

**Bài 2**

- Mức 1: tần suất `b: 3, a: 2, c: 2`; thứ tự theo tần suất giảm dần: `bbbaacc` (hoặc `bbbcca a`, tùy cách phá hòa).
- Mức 2: vị trí lấp lần lượt: `0, 2, 4, 6, 1, 3, 5`.
- Mức 3: ba chữ `b` vào vị trí `0, 2, 4`; hai chữ `a` vào `6, 1`; hai chữ `c` vào `3, 5` ⇒ `b a b c b c a`.

**Bài 3**

- Mức 1: điều kiện khả thi đã học ở mục 4.
- Mức 2: khi `i >= n` thì đặt `i = 1`.
- Mức 3: `if max(cnt.values()) > (n + 1) // 2: return ""` và `if i >= n: i = 1`.

**Bài 4**

- Mức 1: dùng `heapify` rồi vòng `while heap`.
- Mức 2: biến `prev` giữ `(đếm âm, ký tự)` của ký tự lượt trước; đầu mỗi lượt sau khi lấy ký tự mới thì `heappush(heap, prev)`.
- Mức 3: `if c + 1 < 0: prev = (c + 1, ch)`; cuối cùng `return "".join(res) if prev is None else ""`.

**Bài 5**

- Mức 1: giống heap giữ chân nhưng ký tự vừa dùng phải **nguội `k` bước** trước khi được dùng lại: dùng một hàng đợi `deque` chờ.
- Mức 2: sau khi dùng một ký tự ở bước `i`, đưa `(đếm còn lại, ký tự)` vào hàng đợi; khi hàng đợi có độ dài `k` thì lấy phần tử đầu trả vào heap (nếu còn đếm).
- Mức 3: nếu heap rỗng mà chưa đủ độ dài chuỗi thì `""`; hợp lệ khi `len(res) == len(s)`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [36]:
# Bài 1
answers1 = {"aabbb": True, "aaabbbc": True, "abcabcabc": True, "aaaabc": False}
for s, expected in answers1.items():
    assert feasible_formula(s) == expected, s
    assert is_valid_reorg(s, Solution().reorganizeString(s))
    print(f"{s}: n = {len(s)}, ngưỡng = {(len(s) + 1) // 2}, max = {max(Counter(s).values())} -> xếp được = {expected}, Solution trả {Solution().reorganizeString(s)!r}")

# Bài 2
s = "aabbbcc"
n = len(s)
order = list(range(0, n, 2)) + list(range(1, n, 2))
print("Bài 2: thứ tự vị trí lấp:", order, "| kết quả:", Solution().reorganizeString(s))
assert is_valid_reorg(s, Solution().reorganizeString(s))

aabbb: n = 5, ngưỡng = 3, max = 3 -> xếp được = True, Solution trả 'babab'
aaabbbc: n = 7, ngưỡng = 4, max = 3 -> xếp được = True, Solution trả 'ababacb'
abcabcabc: n = 9, ngưỡng = 5, max = 3 -> xếp được = True, Solution trả 'abacacbcb'
aaaabc: n = 6, ngưỡng = 3, max = 4 -> xếp được = False, Solution trả ''
Bài 2: thứ tự vị trí lấp: [0, 2, 4, 6, 1, 3, 5] | kết quả: babcbca


In [37]:
# Bài 3
def reorganize_fill_answer(s):
    n = len(s)
    cnt = Counter(s)
    if max(cnt.values()) > (n + 1) // 2:
        return ""
    res = [None] * n
    i = 0
    for ch, c in cnt.most_common():
        for _ in range(c):
            if i >= n:
                i = 1
            res[i] = ch
            i += 2
    return "".join(res)

# Bài 4
def reorganize_heap_answer(s):
    heap = [(-c, ch) for ch, c in Counter(s).items()]
    heapify(heap)
    res = []
    prev = None
    while heap:
        c, ch = heappop(heap)
        res.append(ch)
        if prev is not None:
            heappush(heap, prev)
            prev = None
        if c + 1 < 0:
            prev = (c + 1, ch)
    return "".join(res) if prev is None else ""

# Bài 5
from collections import deque

def reorganize_k_apart_answer(s, k):
    if k <= 1:
        return s
    heap = [(-c, ch) for ch, c in Counter(s).items()]
    heapify(heap)
    wait = deque()
    res = []
    while heap:
        c, ch = heappop(heap)
        res.append(ch)
        wait.append((c + 1, ch))
        if len(wait) >= k:
            c2, ch2 = wait.popleft()
            if c2 < 0:
                heappush(heap, (c2, ch2))
    return "".join(res) if len(res) == len(s) else ""

def is_valid_k(s, out, k, feasible):
    if not feasible:
        return out == ""
    if sorted(out) != sorted(s):
        return False
    last = {}
    for i, ch in enumerate(out):
        if ch in last and i - last[ch] < k:
            return False
        last[ch] = i
    return True

def k_apart_bruteforce(s, k):
    for p in set(permutations(s)):
        last, good = {}, True
        for i, ch in enumerate(p):
            if ch in last and i - last[ch] < k:
                good = False
                break
            last[ch] = i
        if good:
            return True
    return False

rng = random.Random(1999)
for _ in range(300):
    n = rng.randint(1, 7)
    s = "".join(rng.choice("abc") for _ in range(n))
    assert is_valid_reorg(s, reorganize_fill_answer(s))
    assert is_valid_reorg(s, reorganize_heap_answer(s))
    for k in (1, 2, 3):
        feas = k_apart_bruteforce(s, k)
        out = reorganize_k_apart_answer(s, k)
        # nếu thuật toán trả đáp án thì phải hợp lệ; nếu trả "" mà thực ra khả thi thì ghi nhận
        if out:
            assert is_valid_k(s, out, k, True), (s, k, out)
print("Bài 3, 4: OK")
mismatch = 0
for n in range(1, 8):
    for tup in product("abc", repeat=n):
        s = "".join(tup)
        for k in (2, 3):
            if (reorganize_k_apart_answer(s, k) != "") != k_apart_bruteforce(s, k):
                mismatch += 1
print("Bài 5: số chuỗi (n <= 7, k = 2, 3) mà tính khả thi khác vét cạn:", mismatch)
assert mismatch == 0

Bài 3, 4: OK


Bài 5: số chuỗi (n <= 7, k = 2, 3) mà tính khả thi khác vét cạn: 0


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (lấp vị trí chẵn rồi lẻ theo tần suất):**

```python
class Solution:
    def reorganizeString(self, s: str) -> str:
        n = len(s)
        cnt = Counter(s)                                  # from collections import Counter
        if max(cnt.values()) > (n + 1) // 2:              # quá nhiều một ký tự -> không thể
            return ""
        res = [None] * n
        i = 0
        for ch, c in cnt.most_common():                   # nhiều nhất trước
            for _ in range(c):
                if i >= n:
                    i = 1                                 # hết vị trí chẵn -> sang vị trí lẻ
                res[i] = ch
                i += 2
        return "".join(res)
```

**Câu thần chú:** *"Nhiều nhất không quá nửa làm tròn lên; nhiều nhất xếp trước, lấp chẵn rồi lẻ."* (hoặc heap: *"lấy nhiều nhất, giữ chân một lượt"*).

**Checklist trước khi submit:**

- [ ] Kiểm tra `max_count <= (n + 1) // 2`, nếu không thì trả `""` (không trả chuỗi sai).
- [ ] Ký tự đặt theo **tần suất giảm dần** (không phải thứ tự chữ cái).
- [ ] Heap: dùng số âm; **giữ chân** ký tự vừa dùng một lượt, không trả lại ngay.
- [ ] Hiểu rằng đề nhận **mọi** đáp án hợp lệ: test bằng hàm kiểm tra tính hợp lệ, không so chuỗi cứng.
- [ ] Đã thử: `"a"`, `"aa"`, `"aab"`, `"aaab"`, ký tự đúng bằng ngưỡng, `n = 500`.
- [ ] Đã đối chiếu tính khả thi với vét cạn hoán vị trên chuỗi ngắn.
- [ ] Độ phức tạp: `O(n)` thời gian, `O(n)` bộ nhớ.